# 准备工作

In [8]:
# 设置路径并检查文件大小

from pathlib import Path
import polars as pl

PROJECT_DIR = Path.cwd().parent
RAW_DIR = PROJECT_DIR / "raw_data"
CLEAN_DIR = PROJECT_DIR / "cleaned_data"

STUDENT_VLE_PATH = RAW_DIR / "studentVle.csv"

print("File exists:", STUDENT_VLE_PATH.exists())
print(
    "File size:",
    round(STUDENT_VLE_PATH.stat().st_size / (1024 ** 2), 2),
    "MB"
)

File exists: True
File size: 432.81 MB


In [13]:
# 读取 cleaned reference tables

vle = pl.read_parquet(
    CLEAN_DIR / "vle_clean.parquet"
)

student_info = pl.read_parquet(
    CLEAN_DIR / "student_info_clean.parquet"
)

print(vle.shape)
print(student_info.shape)

(6364, 6)
(32593, 12)


# 开始检查

In [9]:
# 用 lazy loading 检查 schema

student_vle_lazy = pl.scan_csv(
    STUDENT_VLE_PATH,
    schema_overrides={
        "code_module": pl.String,
        "code_presentation": pl.String,
        "id_student": pl.Int32,
        "id_site": pl.Int32,
        "date": pl.Int16,
        "sum_click": pl.Int32
    }
)

student_vle_lazy.collect_schema()

Schema([('code_module', String),
        ('code_presentation', String),
        ('id_student', Int32),
        ('id_site', Int32),
        ('date', Int16),
        ('sum_click', Int32)])

schema 完全正确，尤其是 date 用 Int16、sum_click 用 Int32，能明显降低 memory usage

In [10]:
# 先读取前 10 rows 检查内容

student_vle_lazy.head(10).collect()

code_module,code_presentation,id_student,id_site,date,sum_click
str,str,i32,i32,i16,i32
"""AAA""","""2013J""",28400,546652,-10,4
"""AAA""","""2013J""",28400,546652,-10,1
"""AAA""","""2013J""",28400,546652,-10,1
"""AAA""","""2013J""",28400,546614,-10,11
"""AAA""","""2013J""",28400,546714,-10,1
"""AAA""","""2013J""",28400,546652,-10,8
"""AAA""","""2013J""",28400,546876,-10,2
"""AAA""","""2013J""",28400,546688,-10,15
"""AAA""","""2013J""",28400,546662,-10,17


output 正常，而且能看到同一个 student、同一个 id_site、同一个 date 会出现多条 records，所以之后需要 aggregate，不能把它们当 duplicates 直接删除

In [11]:
# 先做一个 compact validation，一次检查：
# total rows
# missing values
# sum_click 是否有 non-positive values
# date range

student_vle_summary = student_vle_lazy.select(
    pl.len().alias("total_rows"),
    pl.col("code_module").null_count().alias("missing_code_module"),
    pl.col("code_presentation").null_count().alias("missing_code_presentation"),
    pl.col("id_student").null_count().alias("missing_id_student"),
    pl.col("id_site").null_count().alias("missing_id_site"),
    pl.col("date").null_count().alias("missing_date"),
    pl.col("sum_click").null_count().alias("missing_sum_click"),
    pl.col("date").min().alias("min_date"),
    pl.col("date").max().alias("max_date"),
    pl.col("sum_click").min().alias("min_sum_click"),
    pl.col("sum_click").max().alias("max_sum_click"),
    (pl.col("sum_click") <= 0).sum().alias("non_positive_click_rows")
).collect()

student_vle_summary

total_rows,missing_code_module,missing_code_presentation,missing_id_student,missing_id_site,missing_date,missing_sum_click,min_date,max_date,min_sum_click,max_sum_click,non_positive_click_rows
u32,u32,u32,u32,u32,u32,u32,i16,i16,i32,i32,u32
10655280,0,0,0,0,0,0,-25,269,1,6977,0


这个 summary 很干净：

10,655,280 rows

所有 columns 都没有 missing values

sum_click 全部为正数，range 是 1–6977

date range 是 -25–269

negative date 表示 pre-course activity，不是 error

max_date = 269 与最长的 module_presentation_length = 269 一致

所以目前不需要删除或修改任何 rows

In [14]:
# 做一个compact referential integrity check， 确认：

# 每个 id_site 都能在 vle 找到
# 每个 student–module–presentation 都能在 student_info 找到

student_vle_integrity = pl.DataFrame({
    "check": [
        "studentVle → vle",
        "studentVle → student_info"
    ],
    "unmatched_rows": [
        (
            student_vle_lazy
            .join(
                vle.lazy().select(
                    [
                        "id_site",
                        "code_module",
                        "code_presentation"
                    ]
                ),
                on=[
                    "id_site",
                    "code_module",
                    "code_presentation"
                ],
                how="anti"
            )
            .select(pl.len())
            .collect()
            .item()
        ),
        (
            student_vle_lazy
            .join(
                student_info.lazy().select(
                    [
                        "code_module",
                        "code_presentation",
                        "id_student"
                    ]
                ),
                on=[
                    "code_module",
                    "code_presentation",
                    "id_student"
                ],
                how="anti"
            )
            .select(pl.len())
            .collect()
            .item()
        )
    ]
})

student_vle_integrity

check,unmatched_rows
str,i64
"""studentVle → vle""",0
"""studentVle → student_info""",0


两个 unmatched_rows 都是 0，说明：

studentVle 中每个 id_site 都能在 vle 找到

每个 student–module–presentation record 都能在 student_info 找到

所以 referential integrity check 通过

## aggregate 成更小的 daily-level VLE dataset

In [ ]:
# 检查 aggregated dataset 的 structure

student_vle_daily_lazy = (
    student_vle_lazy
    .group_by(
        [
            "code_module",
            "code_presentation",
            "id_student",
            "date"
        ]
    )
    .agg(
        pl.col("sum_click").sum().alias("daily_clicks"),
        pl.col("id_site").n_unique().alias("daily_unique_sites"),
        pl.len().alias("daily_interaction_records")
    )
)

student_vle_daily_lazy.collect_schema()

Schema([('code_module', String),
        ('code_presentation', String),
        ('id_student', Int32),
        ('date', Int16),
        ('daily_clicks', Int32),
        ('daily_unique_sites', UInt32),
        ('daily_interaction_records', UInt32)])

schema 正常：

grouping keys 都保留

daily_clicks 是 Int32

daily_unique_sites 和 daily_interaction_records 是 count 类型

现在还没有真正把全部结果读进 memory

In [ ]:
# 直接把 aggregated data 写成 Parquet，避免先完整 collect()

student_vle_daily_lazy.sink_parquet(
    CLEAN_DIR / "student_vle_daily.parquet",
    compression="zstd"
)

print("student_vle_daily.parquet saved successfully.")

student_vle_daily.parquet saved successfully.


说明 daily-level aggregation 已经成功保存，而且没有把完整结果先加载进 memory

In [17]:
# 检查保存后的文件大小和 row count

STUDENT_VLE_DAILY_PATH = CLEAN_DIR / "student_vle_daily.parquet"

daily_file_size_mb = (
    STUDENT_VLE_DAILY_PATH.stat().st_size / (1024 ** 2)
)

daily_summary = (
    pl.scan_parquet(STUDENT_VLE_DAILY_PATH)
    .select(
        pl.len().alias("total_daily_rows"),
        pl.col("daily_clicks").min().alias("min_daily_clicks"),
        pl.col("daily_clicks").max().alias("max_daily_clicks"),
        pl.col("daily_unique_sites").min().alias("min_unique_sites"),
        pl.col("daily_unique_sites").max().alias("max_unique_sites")
    )
    .collect()
)

print("File size:", round(daily_file_size_mb, 2), "MB")
daily_summary

File size: 9.41 MB


total_daily_rows,min_daily_clicks,max_daily_clicks,min_unique_sites,max_unique_sites
u32,i32,i32,u32,u32
1808119,1,6988,1,235


aggregation 结果非常合理：

rows 从 10,655,280 降到 1,808,119

文件从 432.81 MB 降到 9.41 MB

每个 student-day 至少有 1 click 和 1 unique site

max_daily_clicks = 6988 虽然较高，但目前不能直接视为异常

max_unique_sites = 235 也可能代表某日访问了大量不同 VLE resources

In [18]:
# 确认 aggregation 没有丢失 clicks，并且 daily-level key 没有重复

student_vle_daily_lazy = pl.scan_parquet(
    CLEAN_DIR / "student_vle_daily.parquet"
)

aggregation_validation = pl.DataFrame({
    "check": [
        "Raw total clicks",
        "Daily aggregated total clicks",
        "Duplicate daily keys"
    ],
    "value": [
        (
            student_vle_lazy
            .select(pl.col("sum_click").sum())
            .collect()
            .item()
        ),
        (
            student_vle_daily_lazy
            .select(pl.col("daily_clicks").sum())
            .collect()
            .item()
        ),
        (
            student_vle_daily_lazy
            .group_by([
                "code_module",
                "code_presentation",
                "id_student",
                "date"
            ])
            .len()
            .filter(pl.col("len") > 1)
            .select(pl.len())
            .collect()
            .item()
        )
    ]
})

aggregation_validation

check,value
str,i64
"""Raw total clicks""",39605099
"""Daily aggregated total clicks""",39605099
"""Duplicate daily keys""",0


validation 全部通过：

Raw total clicks = 39,605,099

Daily aggregated total clicks = 39,605,099

Duplicate daily keys = 0

说明 aggregation 没有丢失 clicks，也没有产生重复的 student-day records。student_vle_daily.parquet 现在可以作为后续 temporal feature construction 的基础数据。

## temporal feature construction

In [19]:
# 检查每个 module–presentation 的 daily coverage

daily_coverage = (
    student_vle_daily_lazy
    .group_by([
        "code_module",
        "code_presentation"
    ])
    .agg(
        pl.col("id_student").n_unique().alias("n_students"),
        pl.len().alias("n_student_days"),
        pl.col("date").min().alias("min_date"),
        pl.col("date").max().alias("max_date"),
        pl.col("daily_clicks").sum().alias("total_clicks")
    )
    .sort([
        "code_module",
        "code_presentation"
    ])
    .collect()
)

daily_coverage

code_module,code_presentation,n_students,n_student_days,min_date,max_date,total_clicks
str,str,u32,u32,i16,i16,i32
"""AAA""","""2013J""",378,34072,-10,268,648494
"""AAA""","""2014J""",357,30703,-24,269,598158
"""BBB""","""2013B""",1537,78647,-9,240,1347911
"""BBB""","""2013J""",1870,93548,-23,268,1378656
"""BBB""","""2014B""",1294,56100,-9,234,833865
…,…,…,…,…,…,…
"""FFF""","""2014B""",1363,96201,-18,241,2975619
"""FFF""","""2014J""",2121,167164,-18,269,5281809
"""GGG""","""2013J""",895,32352,-16,261,509091


coverage 看起来正常：

每个 presentation 的 max_date 都与对应的 module_presentation_length 接近

negative min_date 代表 course start 前已经发生的 VLE activity

不同 module 的 n_students、n_student_days 和 total_clicks 差异较大，这是正常的 course-level variation

In [20]:
# 确认没有任何 VLE activity 超过对应课程长度

courses = pl.read_parquet(
    CLEAN_DIR / "courses_clean.parquet"
)

date_range_validation = (
    student_vle_daily_lazy
    .join(
        courses.lazy(),
        on=[
            "code_module",
            "code_presentation"
        ],
        how="left"
    )
    .select(
        pl.len().alias("total_daily_rows"),
        (
            pl.col("date") >
            pl.col("module_presentation_length")
        ).sum().alias("rows_after_module_end"),
        (
            pl.col("date") <
            -30
        ).sum().alias("rows_before_day_minus_30"),
        (
            pl.col("module_presentation_length").is_null()
        ).sum().alias("missing_course_match")
    )
    .collect()
)

date_range_validation

total_daily_rows,rows_after_module_end,rows_before_day_minus_30,missing_course_match
u32,u32,u32,u32
1808119,0,0,0


这一步也全部通过：

1,808,119 条 daily records

rows_after_module_end = 0

rows_before_day_minus_30 = 0

missing_course_match = 0

说明 student_vle_daily.parquet 的 date range 和 course mapping 都没有问题。到这里，studentVle 的基础清洗和 daily aggregation 已经完成